In [38]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00


In [39]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier



In [22]:
df = pd.read_csv("cropdata_updated.csv")
print(f"Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head(10)

Raw Dataset Shape: 16411 rows, 7 columns



,crop ID,soil_type,Seedling Stage,MOI,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1
5,Wheat,Black Soil,Germination,6,30,65.0,1
6,Wheat,Black Soil,Germination,7,31,62.0,1
7,Wheat,Black Soil,Germination,8,32,59.0,1
8,Wheat,Black Soil,Germination,9,33,56.0,1
9,Wheat,Black Soil,Germination,10,34,53.0,1


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16411 entries, 0 to 16410
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crop ID         16411 non-null  object 
 1   soil_type       16411 non-null  object 
 2   Seedling Stage  16411 non-null  object 
 3   MOI             16411 non-null  int64  
 4   temp            16411 non-null  int64  
 5   humidity        16411 non-null  float64
 6   result          16411 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 897.6+ KB


In [24]:
df.drop_duplicates(inplace=True)
df.shape

(16283, 7)

In [25]:
df = df[df["result"] != 2]
df.shape

(15161, 7)

In [26]:
target_counts = df['result'].value_counts().sort_index()
target_pct = df['result'].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage (%)': target_pct.round(2),
    'Description': [
        '0: No Irrigation',
        '1: Standard Irrigation',
    ]
})
target_summary

,Count,Percentage (%),Description
result,,,
0,8934,58.93,0: No Irrigation
1,6227,41.07,1: Standard Irrigation


In [27]:
df.rename(columns={
    "crop ID": "crop_name",
    "soil_type": "soil_type",
    "Seedling Stage": "seedling_stage",
    "MOI": "moi",
    "temp": "temp",
    "humidity": "humidity",
    "result": "result",
}, inplace=True)
df.head()

,crop_name,soil_type,seedling_stage,moi,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1


In [28]:
numerical_cols = ['moi', 'temp', 'humidity']

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"The column {col} has {len(outliers)} outliers that has a percentage of {round((len(outliers) / len(df)) * 100, 2)}%")



The column moi has 0 outliers that has a percentage of 0.0%
The column temp has 0 outliers that has a percentage of 0.0%
The column humidity has 0 outliers that has a percentage of 0.0%


In [29]:
GROWTH_STAGE_ORDER_MAP = {
    "Germination": 1,
    "Seedling Stage": 2,
    "Vegetative Growth / Root or Tuber Development": 3,
    "Flowering": 4,
    "Pollination": 5,
    "Fruit/Grain/Bulb Formation": 6,
    "Maturation": 7,
    "Harvest": 8,
}
df["growth_stage_order"] = (df["seedling_stage"].map(GROWTH_STAGE_ORDER_MAP).fillna(0).astype(int))

In [30]:
df["moi_temp_ratio"] = (df["moi"] / df["temp"].replace(0, np.nan)).round(4)
df["moi_humidity_index"] = ((df["moi"] * df["humidity"]) / 100.0).round(4)
df.shape

(15161, 10)

In [ ]:
df.to_csv("./data/cleaned_data.csv", index=False)

In [48]:
numerical_cols = ["moi", "temp", "humidity", "growth_stage_order", "moi_temp_ratio", "moi_humidity_index"]
categorial_cols = [ "crop_name","soil_type", "seedling_stage"]
feature_cols = categorial_cols + numerical_cols
X = df[feature_cols]
X= X.drop(columns="crop_name")
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [49]:
categorial_cols = [ "soil_type", "seedling_stage"]
processor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),["moi", "temp", "humidity", "moi_temp_ratio", "moi_humidity_index"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorial_cols)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [50]:
X_train_transformed = processor.fit_transform(X_train)
X_test_transformed = processor.transform(X_test)
feature_names = processor.get_feature_names_out()

#X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names)
#X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names)

In [51]:
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l2'],
            'solver': ['lbfgs', 'liblinear']
        }

    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5]
        }
    },
    'SVC': {
        'model': SVC(random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto']
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7]
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {'max_depth': [None, 5, 10, 20], 'criterion': ['gini', 'entropy']},
        'use_scaler': False}
    ,
    'AdaBoost': {
        'model': AdaBoostClassifier(random_state=42),
        'params': {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1, 1.0]},
        'use_scaler': False}
    ,
    'CatBoost': {
        'model': CatBoostClassifier(random_state=42, verbose=0),
        'params': {'iterations': [100, 200], 'learning_rate': [0.03, 0.1], 'depth': [4, 6]},
        'use_scaler': False
    }
}

In [52]:
results = []
X_tr = X_train_transformed
X_te =X_test_transformed

for name, config in models_config.items():


    base_model = config['model']
    base_model.fit(X_tr, y_train)

    y_pred = base_model.predict(X_te)
    acc_before = accuracy_score(y_test, y_pred)
    print(f"=== {name} ===")
    print(f"Classification Report Before Grid Search on  test data:")
    print(classification_report(y_test, y_pred))


    grid = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    grid.fit(X_tr, y_train)
    y_pred = grid.best_estimator_.predict(X_te)
    acc_after = accuracy_score(y_test,y_pred)

    print(f"=== {name} ===")
    print(f"Classification Report After Grid Search on test data :")
    print(classification_report(y_test, y_pred))


    results.append({
        'Model': name,
        'Accuracy (Before)': f"{acc_before:.4f}",
        'Accuracy (After)': f"{acc_after:.4f}",
        'Best Params': grid.best_params_
    })


df_results = pd.DataFrame(results)
print("=== Comparison Before and After GridSearchCV ===")
print(df_results[['Model', 'Accuracy (Before)', 'Accuracy (After)']])

print("\n=== Best Hyperparameters ===")
for res in results:
    print(f"{res['Model']}: {res['Best Params']}")

=== Logistic Regression ===
Classification Report Before Grid Search on  test data:
              precision    recall  f1-score   support

           0       0.93      0.96      0.95      1787
           1       0.94      0.90      0.92      1246

    accuracy                           0.94      3033
   macro avg       0.94      0.93      0.93      3033
weighted avg       0.94      0.94      0.94      3033

=== Logistic Regression ===
Classification Report After Grid Search on test data :
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1787
           1       0.95      0.90      0.92      1246

    accuracy                           0.94      3033
   macro avg       0.94      0.93      0.94      3033
weighted avg       0.94      0.94      0.94      3033

=== Random Forest ===
Classification Report Before Grid Search on  test data:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98   